Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu124/


In [ ]:
!pip install paddleocr "paddlex[ocr]"

In [3]:
!pip install "langchain>=0.1,<1.0"

In [ ]:


pdf_path = "/content/drive/MyDrive/pdf_folder/sperated_small.pdf"
input_file = pdf_path  


import os
print("OK" if os.path.exists(pdf_path) else "?? ??:", pdf_path)


OK /content/drive/MyDrive/pdf_folder/sperated_small.pdf


In [ ]:
import json
from pathlib import Path
from paddleocr import PPStructureV3
output_dir = Path("/content/drive/MyDrive/pdf_folder/ocr_results")
output_dir.mkdir(parents=True, exist_ok=True)

input_file = Path("/content/drive/MyDrive/pdf_folder/main.pdf") 

pipeline = PPStructureV3(lang="de", device="gpu")
output = pipeline.predict(input=str(input_file))

markdown_list = []
footnote_data = []
markdown_images = [] 

for res in output:
    md_info = res.markdown
    markdown_list.append(md_info)
    markdown_images.append(md_info.get("markdown_images", {})) 
    
    blocks = res.json["res"]["parsing_res_list"]

    number = next(
        (b["block_content"] for b in blocks if b["block_label"] == "number"),
        None,
    )
    footnotes = [
        b["block_content"]
        for b in blocks
        if b["block_label"] == "footnote"
    ]
    if footnotes:
        footnote_data.append({"number": number, "footnote": footnotes})


markdown_result = pipeline.concatenate_markdown_pages(markdown_list)
markdown_texts = markdown_result.markdown["markdown_texts"]
stem = Path(input_file).stem
mkd_path = output_dir / f"{stem}.md"
with open(mkd_path, "w", encoding="utf-8") as f:
    f.write(markdown_texts)
print(f"{mkd_path}")


json_path = output_dir / f"{stem}_footnotes.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(footnote_data, f, ensure_ascii=False, indent=4)
print(f"{json_path}")

for item in markdown_images:
    if item:
        for path, image in item.items():
            file_path = output_dir / path
            file_path.parent.mkdir(parents=True, exist_ok=True)
            image.save(file_path)

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-DocBlockLayout', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocBlockLayout`.
Creating model: ('PP-DocLayout_plus-L', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayout_plus-L`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('

/content/drive/MyDrive/pdf_folder/ocr_results/main.md
/content/drive/MyDrive/pdf_folder/ocr_results/main_footnotes.json
